# Jaguar Re-identification Training Pipeline

This notebook demonstrates the complete re-identification pipeline:
1. Configuration setup
2. Data loading from FiftyOne/disk/HuggingFace
3. Model training with backbone feature extraction
4. Evaluation
5. Results export

Based on `jaguars.reidentification.pipeline`

## Setup and Imports

In [1]:
import sys
from pathlib import Path
import logging

# Add src to path if needed
project_root = Path.cwd().parent / "camera-trap-footage"
sys.path.insert(0, str(project_root / "src"))

from jaguars.common.logging_utils import setup_logger
from jaguars.reidentification.config import ReidentificationConfig, get_default_config
from jaguars.reidentification.training.train import run_processing as run_training
from jaguars.reidentification.evaluation.evaluation import run_processing as run_evaluation
from jaguars.reidentification.export_results import run_processing as run_export

# Setup logging
logger = setup_logger("reidentification_notebook", level=logging.INFO)
print("✓ Imports successful")

✓ Imports successful


## Set Random Seeds for Reproducibility

In [2]:
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    print(f"✓ Random seeds set to {SEED} (GPU available)")
else:
    print(f"✓ Random seeds set to {SEED} (CPU only)")

✓ Random seeds set to 42 (GPU available)


## Configuration Setup

Configure the pipeline parameters. You can modify these based on your needs.

In [ ]:
# Get default configuration
config = get_default_config()

# =============================================================================
# Dataset Configuration
# =============================================================================
config.dataset.source = "fiftyone"  # Options: "fiftyone", "disk", "huggingface"
config.dataset.fo_dataset_name = "JID_Master_Dataset"  # FiftyOne dataset name

# Split configuration - IMPORTANT: Use the split field created by split.py
config.dataset.fo_split_field = "closed_set_split"  # Options: "closed_set_split", "open_set_split"

# Patch configuration - set to None for full-image training, or "sam3_segmentations" for detection-level
config.dataset.fo_patches_field = "sam3_segmentations"

# Label field configuration
config.dataset.fo_label_field = "ground_truth"  # Field containing jaguar ID labels

# Precomputed embeddings (speeds up training by skipping backbone extraction)
config.dataset.fo_embeddings_field = "embeddings_BVRA_MegaDescriptor_L_384"

# =============================================================================
# Training Configuration - Optimized for Re-ID
# =============================================================================
config.training.batch_size = 32
config.training.num_epochs = 50
config.training.learning_rate = 1e-4
config.training.weight_decay = 1e-4
config.training.save_dir = Path("data/models/reidentification")

# Loss Function - ArcFace is recommended for re-identification
# Options: "arcface", "subcenter_arcface", "arcface_triplet", "triplet", 
#          "cross_entropy", "focal", "label_smoothing"
config.training.loss_name = "arcface"

# Triplet loss settings (used when loss_name includes "triplet")
config.training.triplet_margin = 0.3
config.training.triplet_mining = "hard"  # Options: "all", "hard", "semi-hard"
config.training.triplet_weight = 0.5  # Weight when combined with ArcFace

# PK Sampler (recommended for triplet losses)
config.training.use_pk_sampler = False  # Set True for triplet/arcface_triplet
config.training.pk_p = 8  # Number of classes per batch
config.training.pk_k = 4  # Samples per class (batch_size = pk_p * pk_k = 32)

# Scheduler
config.training.scheduler_type = "cosine"  # Options: "cosine", "reduce_on_plateau", "step"
config.training.scheduler_patience = 5
config.training.scheduler_factor = 0.5

# Early stopping
config.training.early_stopping_patience = 10
config.training.early_stopping_metric = "val_map"
config.training.early_stopping_mode = "max"

# =============================================================================
# Backbone Configuration
# =============================================================================
config.backbone.name = "BVRA/MegaDescriptor-L-384"  # MegaDescriptor Large (1536-dim)
config.backbone.pretrained = True
config.backbone.embedding_dim = 1536  # Output from MegaDescriptor-L

# =============================================================================
# Model Configuration - ArcFace Re-ID Head
# =============================================================================
config.model.embedding_dim = 256  # Projected embedding dimension
config.model.hidden_dim = 512  # Hidden layer in projection network
config.model.dropout = 0.3  # Dropout for regularization
config.model.arcface_margin = 0.5  # Angular margin (0.3-0.7 typical)
config.model.arcface_scale = 64.0  # Feature scale (30-64 typical)

# =============================================================================
# Wandb Configuration
# =============================================================================
config.wandb.enabled = True  # Enable for experiment tracking
config.wandb.entity = "jaguars"
config.wandb.project = "camera-trap-reidentification"
config.wandb.run_name = f"arcface_mega_l384_{SEED}"
config.wandb.tags = ["arcface", "megadescriptor", "closed_set"]

# =============================================================================
# Evaluation Configuration  
# =============================================================================
config.evaluation.save_embeddings = True
config.evaluation.save_predictions = True
config.evaluation.add_to_fiftyone = False
config.evaluation.output_dir = Path("data/results/reidentification")

# Runtime
config.verbose = True
config.seed = SEED

# =============================================================================
# Print Configuration Summary
# =============================================================================
print("=" * 70)
print("CONFIGURATION SUMMARY")
print("=" * 70)
print(f"\n📁 Dataset:")
print(f"  Source: {config.dataset.source}")
print(f"  FiftyOne dataset: {config.dataset.fo_dataset_name}")
print(f"  Split field: {config.dataset.fo_split_field}")
print(f"  Patches field: {config.dataset.fo_patches_field or 'None (sample-level)'}")
print(f"  Label field: {config.dataset.fo_label_field}")
print(f"\n🧠 Model:")
print(f"  Backbone: {config.backbone.name}")
print(f"  Embedding dim: {config.model.embedding_dim}")
print(f"  Hidden dim: {config.model.hidden_dim}")
print(f"  ArcFace margin: {config.model.arcface_margin}")
print(f"  ArcFace scale: {config.model.arcface_scale}")
print(f"\n⚡ Training:")
print(f"  Loss: {config.training.loss_name}")
print(f"  Batch size: {config.training.batch_size}")
print(f"  Epochs: {config.training.num_epochs}")
print(f"  Learning rate: {config.training.learning_rate}")
print(f"  Scheduler: {config.training.scheduler_type}")
if "triplet" in config.training.loss_name:
    print(f"  Triplet mining: {config.training.triplet_mining}")
    print(f"  Triplet margin: {config.training.triplet_margin}")
if config.training.use_pk_sampler:
    print(f"  PK Sampler: P={config.training.pk_p}, K={config.training.pk_k}")
print(f"\n📊 Wandb: {'Enabled' if config.wandb.enabled else 'Disabled'}")
if config.wandb.enabled:
    print(f"  Project: {config.wandb.project}")
    print(f"  Run name: {config.wandb.run_name}")
print("=" * 70)

CONFIGURATION SUMMARY

📁 Dataset:
  Source: fiftyone
  FiftyOne dataset: JID_Master_Dataset
  Split field: closed_set_split
  Patches field: sam3_segmentations
  Label field: ground_truth

🧠 Model:
  Backbone: BVRA/MegaDescriptor-L-384
  Embedding dim: 256
  Hidden dim: 512
  ArcFace margin: 0.5
  ArcFace scale: 64.0

⚡ Training:
  Loss: arcface
  Batch size: 32
  Epochs: 50
  Learning rate: 0.0001
  Scheduler: cosine

📊 Wandb: Enabled
  Project: camera-trap-reidentification
  Run name: arcface_mega_l384_42


## Load Dataset from Disk (Optional)

If you want to load a specific dataset variant from disk into FiftyOne, run this cell.

In [4]:
# Configuration for loading from disk
LOAD_FROM_DISK = False  # Set to True to load from disk
DATASET_VARIANT = "segmented_deduplicated"  # Options: master, segmented_deduplicated, segmented, not_segmented_deduplicated, not_segmented
DATASET_DISK_PATH = Path("../data/intermediate/v1/fo_jaguars/exports") / DATASET_VARIANT
IMPORTED_DATASET_NAME = f"JID_{DATASET_VARIANT}"
OVERWRITE_IF_EXISTS = False

if LOAD_FROM_DISK:
    import fiftyone as fo
    
    print("=" * 70)
    print(f"Loading dataset variant '{DATASET_VARIANT}' from disk")
    print("=" * 70)
    
    # Check if dataset already exists
    if fo.dataset_exists(IMPORTED_DATASET_NAME):
        if OVERWRITE_IF_EXISTS:
            print(f"Deleting existing dataset '{IMPORTED_DATASET_NAME}'...")
            fo.delete_dataset(IMPORTED_DATASET_NAME)
        else:
            print(f"✓ Dataset '{IMPORTED_DATASET_NAME}' already exists in FiftyOne")
            print("Set OVERWRITE_IF_EXISTS=True to reload from disk")
            imported_dataset = fo.load_dataset(IMPORTED_DATASET_NAME)
            print(f"  Total samples: {len(imported_dataset)}")
    
    if not fo.dataset_exists(IMPORTED_DATASET_NAME):
        if not DATASET_DISK_PATH.exists():
            print(f"⚠ Error: Dataset not found at {DATASET_DISK_PATH}")
            print("Make sure you've run the export step in the ingestion pipeline first.")
        else:
            print(f"Loading from: {DATASET_DISK_PATH}")
            imported_dataset = fo.Dataset.from_dir(
                dataset_dir=str(DATASET_DISK_PATH),
                dataset_type=fo.types.FiftyOneDataset,
                name=IMPORTED_DATASET_NAME,
            )
            print(f"✓ Dataset loaded and saved as '{IMPORTED_DATASET_NAME}'")
            print(f"  Total samples: {len(imported_dataset)}")
            
            # Update config to use this dataset
            config.dataset.source = "fiftyone"
            config.dataset.fo_dataset_name = IMPORTED_DATASET_NAME
            print(f"\n✓ Configuration updated to use '{IMPORTED_DATASET_NAME}'")
else:
    print("Skipping disk import. Set LOAD_FROM_DISK=True to import a dataset variant.")

Skipping disk import. Set LOAD_FROM_DISK=True to import a dataset variant.


## Step 1: Training

Train the re-identification model with the configured parameters.

## Understanding the Re-ID Training Pipeline

### Training Flow
```
┌─────────────────────────────────────────────────────────────────────────┐
│  PRE-COMPUTED (Backbone)                                                 │
│  ┌──────────────┐      ┌──────────────────────────────┐                 │
│  │  Input Image │ ───► │  MegaDescriptor-L-384        │ ───► 1536-dim   │
│  │  (384×384)   │      │  (frozen backbone)           │      embedding  │
│  └──────────────┘      └──────────────────────────────┘                 │
└─────────────────────────────────────────────────────────────────────────┘
                                      │
                                      ▼
┌─────────────────────────────────────────────────────────────────────────┐
│  LEARNED (Projection Head + ArcFace)                                     │
│  ┌──────────────┐      ┌──────────────┐      ┌──────────────┐          │
│  │  1536-dim    │ ───► │  Projection  │ ───► │  ArcFace     │ ───► CE  │
│  │  backbone    │      │  Network     │      │  Layer       │     Loss │
│  │  embedding   │      │  (256-dim)   │      │  (logits)    │          │
│  └──────────────┘      └──────────────┘      └──────────────┘          │
└─────────────────────────────────────────────────────────────────────────┘
```

### Key Components

1. **Backbone (Pre-computed)**: MegaDescriptor-L-384 extracts 1536-dimensional feature vectors
2. **Projection Network**: Reduces 1536 → 512 → 256 dimensions with BatchNorm + ReLU + Dropout
3. **ArcFace Layer**: Applies angular margin to push embeddings of same identity together
4. **Training Loss**: CrossEntropy on ArcFace logits (classification objective)

### Evaluation
- Extract 256-dim embeddings from projection network (no ArcFace)
- L2-normalize embeddings
- Use cosine similarity for retrieval/ranking
- Compute mAP (mean Average Precision)

### Why ArcFace works for Re-ID
- Creates discriminative embeddings on the hypersphere
- Angular margin separates identities with angular distance
- Works well even with many classes (individual jaguars)
- Learns to distinguish fine-grained differences

In [7]:
print("=" * 70)
print("STEP 1: Training")
print("=" * 70)

training_results = run_training(config=config, verbose=config.verbose)

print("\nTraining completed!")
print(f"Best validation mAP: {training_results.get('best_val_map', 'N/A'):.4f}")
print(f"Best model saved to: {config.training.save_dir / 'best_model.pt'}")

STEP 1: Training
20:02:09 - jid_logger.reidentification.training - INFO - Starting re-identification training...
20:02:09 - jid_logger.reidentification.training - INFO - Dataset source: fiftyone
20:02:09 - jid_logger.reidentification.training - INFO - Backbone: BVRA/MegaDescriptor-L-384
20:02:09 - jid_logger.reidentification.training - INFO - Device: cuda
20:02:20 - jid_logger.reidentification.training - INFO - Resource validation passed


20:02:23 - jid_logger.reidentification.training - INFO - Loading dataset...
20:02:39 - jid_logger.reidentification.training - INFO - Dataset loaded:
20:02:39 - jid_logger.reidentification.training - INFO -   Train: 946 samples
20:02:39 - jid_logger.reidentification.training - INFO -   Val: 120 samples
20:02:39 - jid_logger.reidentification.training - INFO -   Num classes: 76
20:02:39 - jid_logger.reidentification.training - INFO - Using pre-computed embeddings
20:02:40 - jid_logger.reidentification.training - INFO - DataLoaders created:
20:02:40 - jid_logger.reidentification.training - INFO -   Train batches: 30
20:02:40 - jid_logger.reidentification.training - INFO -   Val batches: 4
Model initialized:
  Input dim: 1536
  Hidden dim: 512
  Embedding dim: 256
  Num classes: 76
  ArcFace margin: 0.5
  ArcFace scale: 64.0
  Total parameters: 939,264
20:02:40 - jid_logger.reidentification.training - INFO - Loss: arcface
20:02:40 - jid_logger.reidentification.training - INFO - Training com

20:02:41 - jid_logger.reidentification.training - INFO - Train Loss: 39.9196, Train Acc: 0.00%
20:02:41 - jid_logger.reidentification.training - INFO - Val Loss: 38.5374, Val Acc: 0.00%
20:02:41 - jid_logger.reidentification.training - INFO - Val mAP: 0.1034
20:02:41 - jid_logger.reidentification.training - INFO - Learning Rate: 0.000100
20:02:41 - jid_logger.reidentification.training - INFO - Saved best model to data/models/reidentification/best_model.pt
20:02:41 - jid_logger.reidentification.training - INFO - 
Epoch 2/50


20:02:41 - jid_logger.reidentification.training - INFO - Train Loss: 38.1421, Train Acc: 0.00%


20:02:41 - jid_logger.reidentification.training - INFO - Val Loss: 37.2550, Val Acc: 0.00%
20:02:41 - jid_logger.reidentification.training - INFO - Val mAP: 0.1042
20:02:41 - jid_logger.reidentification.training - INFO - Learning Rate: 0.000100
20:02:41 - jid_logger.reidentification.training - INFO - Saved best model to data/models/reidentification/best_model.pt
20:02:41 - jid_logger.reidentification.training - INFO - 
Epoch 3/50


...

20:02:56 - jid_logger.reidentification.training - INFO - Train Loss: 24.9725, Train Acc: 5.81%
20:02:56 - jid_logger.reidentification.training - INFO - Val Loss: 31.9482, Val Acc: 3.33%
20:02:56 - jid_logger.reidentification.training - INFO - Val mAP: 0.1469
20:02:56 - jid_logger.reidentification.training - INFO - Learning Rate: 0.000001
20:02:56 - jid_logger.reidentification.training - INFO - 
Epoch 49/50


20:02:56 - jid_logger.reidentification.training - INFO - Train Loss: 24.9778, Train Acc: 5.81%
20:02:56 - jid_logger.reidentification.training - INFO - Val Loss: 31.8590, Val Acc: 3.33%
20:02:56 - jid_logger.reidentification.training - INFO - Val mAP: 0.1562
20:02:56 - jid_logger.reidentification.training - INFO - Learning Rate: 0.000000
20:02:56 - jid_logger.reidentification.training - INFO - 
Epoch 50/50


20:02:56 - jid_logger.reidentification.training - INFO - Train Loss: 24.8928, Train Acc: 5.39%
20:02:56 - jid_logger.reidentification.training - INFO - Val Loss: 31.8970, Val Acc: 3.33%
20:02:56 - jid_logger.reidentification.training - INFO - Val mAP: 0.1502
20:02:56 - jid_logger.reidentification.training - INFO - Learning Rate: 0.000000
20:02:56 - jid_logger.reidentification.training - INFO - Saved checkpoint to data/models/reidentification/checkpoint_epoch_50.pt
20:02:56 - jid_logger.reidentification.training - INFO - ======================================================================
20:02:56 - jid_logger.reidentification.training - INFO - Training completed!
20:02:56 - jid_logger.reidentification.training - INFO - Best epoch: 45
20:02:56 - jid_logger.reidentification.training - INFO - Best val_map: 0.1671


epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
lr,████████▇▇▇▇▇▆▆▆▆▅▅▅▄▄▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁
train/acc,▁▁▁▁▁▁▁▁▁▁▁▂▁▂▂▃▄▄▄▄▅▅▆▆▆▅▇▇▇▇▇▇▇██████▇
train/batch_acc,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▁▃▃▃▃▃▁▃▃▄▄▄▄▄▄▄█▄▄▄▄▅▄▄
train/batch_cls_loss,███▇▇▇▆▆▆▆▆▄▅▅▅▆▄▄▄▅▄▅▄▄▄▄▄▄▄▄▃▄▃▃▄▁▃▃▃▅
train/batch_loss,██▇▇▇▇▇▆▆▅▅▅▅▆▅▄▅▅▂▄▃▄▄▁▂▄▃▃▁▃▃▂▁▂▂▃▄▂▃▄
train/loss,█▇▆▆▆▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▁▁▁▁▁▁▁▁▁▁▂▂▄▇▇▇▅▇▇▇▇▇▇▇█▇▇▇▇▇▇▇▇▇▇▇▇▇
val/loss,█▇▆▅▅▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/map,▂▂▁▁▁▁▂▄▃▃▃▄▅▄▄▄▄▅▄▅▆▆▅▇▆▆▇▅▆▆▇▆▆▇▆█▇▇▆▇
epoch,50



Training completed!
Best validation mAP: 0.1671
Best model saved to: data/models/reidentification/best_model.pt


## Step 2: Evaluation

Evaluate the trained model on the test set.

In [5]:
print("=" * 70)
print("STEP 2: Evaluation")
print("=" * 70)

# Use the best model from training
model_path = config.training.save_dir / "best_model.pt"
print(f"Loading model from: {model_path}")

evaluation_results = run_evaluation(
    config=config,
    model_path=model_path,
    verbose=config.verbose
)

print("\nEvaluation completed!")
print(f"Test mAP: {evaluation_results.get('map', 'N/A'):.4f}")
if 'top1_accuracy' in evaluation_results:
    print(f"Top-1 Accuracy: {evaluation_results['top1_accuracy']:.4f}")
if 'top5_accuracy' in evaluation_results:
    print(f"Top-5 Accuracy: {evaluation_results['top5_accuracy']:.4f}")

STEP 2: Evaluation
Loading model from: data/models/reidentification/best_model.pt
19:07:35 - jid_logger.reidentification.evaluation - INFO - Starting evaluation...
19:07:35 - jid_logger.reidentification.evaluation - INFO - Dataset source: fiftyone
19:07:46 - jid_logger.reidentification.evaluation - INFO - Resource validation passed
19:07:46 - jid_logger.reidentification.evaluation - INFO - Loading dataset...
19:07:59 - jid_logger.reidentification.evaluation - INFO - Test set: 105 samples, 31 classes
19:07:59 - jid_logger.reidentification.evaluation - INFO - Using pre-computed embeddings
Model initialized:
  Input dim: 1536
  Hidden dim: 512
  Embedding dim: 256
  Num classes: 76
  ArcFace margin: 0.5
  ArcFace scale: 64.0
  Total parameters: 939,264
19:08:00 - jid_logger.reidentification.evaluation - INFO - Loading model from data/models/reidentification/best_model.pt
19:08:00 - jid_logger.reidentification.evaluation - INFO - Loaded checkpoint from epoch 38
19:08:00 - jid_logger.reiden

## Step 3: Export Results

Export embeddings and predictions to disk and/or FiftyOne.

In [ ]:
print("=" * 70)
print("STEP 3: Export Results")
print("=" * 70)

# Determine export targets based on configuration
export_targets = []
if config.evaluation.save_embeddings or config.evaluation.save_predictions:
    export_targets.append("disk")
if config.evaluation.add_to_fiftyone:
    export_targets.append("fiftyone")

if not export_targets:
    print("No export targets configured. Skipping export.")
else:
    print(f"Export targets: {export_targets}")
    
    export_results = run_export(
        config=config,
        model_path=model_path,
        export_targets=export_targets,
        verbose=config.verbose
    )
    
    print("\nExport completed!")
    if "disk" in export_targets:
        print(f"Results saved to: {config.evaluation.output_dir}")
    if "fiftyone" in export_targets:
        print(f"Results added to FiftyOne dataset: {config.dataset.fo_dataset_name}")

## Summary

Display final results from all pipeline steps.

In [ ]:
print("=" * 70)
print("PIPELINE SUMMARY")
print("=" * 70)

print("\nConfiguration:")
print(f"  Dataset: {config.dataset.source}")
print(f"  Backbone: {config.backbone.name}")
print(f"  Epochs: {config.training.num_epochs}")
print(f"  Batch size: {config.training.batch_size}")

print("\nResults:")
if training_results:
    print(f"  Best validation mAP: {training_results.get('best_val_map', 'N/A'):.4f}")
if evaluation_results:
    print(f"  Test mAP: {evaluation_results.get('map', 'N/A'):.4f}")
    if 'top1_accuracy' in evaluation_results:
        print(f"  Top-1 Accuracy: {evaluation_results['top1_accuracy']:.4f}")
    if 'top5_accuracy' in evaluation_results:
        print(f"  Top-5 Accuracy: {evaluation_results['top5_accuracy']:.4f}")

print("\nOutput locations:")
print(f"  Model: {config.training.save_dir / 'best_model.pt'}")
if config.evaluation.save_embeddings or config.evaluation.save_predictions:
    print(f"  Results: {config.evaluation.output_dir}")

print("\n" + "=" * 70)
print("Pipeline completed successfully!")
print("=" * 70)

## Optional: Visualize Results

Load and visualize embeddings or predictions (if saved).

In [ ]:
# Example: Load and visualize embeddings with t-SNE/UMAP
# This is optional and requires the embeddings to be saved

if config.evaluation.save_embeddings:
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.manifold import TSNE
    
    # Load embeddings (adjust path as needed)
    embeddings_path = config.evaluation.output_dir / "embeddings.npz"
    
    if embeddings_path.exists():
        print(f"Loading embeddings from {embeddings_path}")
        data = np.load(embeddings_path, allow_pickle=True)
        embeddings = data['embeddings']
        labels = data['labels']
        
        print(f"Embeddings shape: {embeddings.shape}")
        print(f"Number of unique individuals: {len(np.unique(labels))}")
        
        # Reduce dimensionality for visualization
        print("Computing t-SNE projection...")
        tsne = TSNE(n_components=2, random_state=SEED)
        embeddings_2d = tsne.fit_transform(embeddings)
        
        # Plot
        plt.figure(figsize=(12, 8))
        scatter = plt.scatter(
            embeddings_2d[:, 0],
            embeddings_2d[:, 1],
            c=labels,
            cmap='tab20',
            alpha=0.6
        )
        plt.colorbar(scatter, label='Individual ID')
        plt.title('Jaguar Re-identification Embeddings (t-SNE)')
        plt.xlabel('t-SNE 1')
        plt.ylabel('t-SNE 2')
        plt.tight_layout()
        plt.show()
    else:
        print(f"Embeddings file not found at {embeddings_path}")
else:
    print("Embeddings were not saved. Set config.evaluation.save_embeddings = True to enable visualization.")

## Run Loss Experiments

Run systematic comparison of different loss functions with Wandb tracking.

In [5]:
# Run loss function experiments
# This will train models with different losses and log results to Wandb

from jaguars.reidentification.experiments import get_loss_experiments
from jaguars.reidentification.training.train import run_processing as run_training

# Get all loss experiments using the notebook config as base
# This preserves your wandb settings, paths, and other configurations
loss_experiments = get_loss_experiments(base_config=config)

print(f"Available loss experiments ({len(loss_experiments)}):")
for exp in loss_experiments:
    print(f"  - {exp.name}: {exp.description}")

# Select which experiments to run (comment out to run all)
EXPERIMENTS_TO_RUN = [
    "loss_random_baseline",
    "loss_arcface",        # Standard ArcFace
    # "loss_arcface_triplet", # ArcFace + Triplet (requires PK sampling)
    "loss_cross_entropy",   # Baseline
]

# Filter to selected experiments
selected_experiments = [
    exp for exp in loss_experiments 
    if exp.name in EXPERIMENTS_TO_RUN
]

print(f"\nSelected {len(selected_experiments)} experiments to run")
print(f"\nUsing configuration from notebook:")
print(f"  - Dataset: {config.dataset.fo_dataset_name}")
print(f"  - Split field: {config.dataset.fo_split_field}")
print(f"  - Backbone: {config.backbone.name}")
print(f"  - Wandb project: {config.wandb.project}")
print(f"  - Wandb entity: {config.wandb.entity}")

# Run experiments
RUN_EXPERIMENTS = True  # Set to True to actually run

if RUN_EXPERIMENTS:
    results = {}
    for i, exp in enumerate(selected_experiments):
        print(f"\n{'='*70}")
        print(f"EXPERIMENT {i+1}/{len(selected_experiments)}: {exp.name}")
        print(f"Description: {exp.description}")
        print(f"{'='*70}")
        
        # Update run name for wandb
        exp.base_config.wandb.run_name = exp.name
        exp.base_config.wandb.tags = exp.tags
        
        try:
            result = run_training(
                config=exp.base_config,
                verbose=True,
            )
            results[exp.name] = result
            print(f"✓ {exp.name} completed: mAP = {result.get('best_val_map', 'N/A'):.4f}")
        except Exception as e:
            print(f"✗ {exp.name} failed: {e}")
            results[exp.name] = {"error": str(e)}
    
    # Summary
    print(f"\n{'='*70}")
    print("EXPERIMENT SUMMARY")
    print(f"{'='*70}")
    for name, result in results.items():
        if "error" in result:
            print(f"  {name}: FAILED - {result['error']}")
        else:
            print(f"  {name}: mAP = {result.get('best_val_map', 'N/A'):.4f}")
else:
    print("\n⚠️ Set RUN_EXPERIMENTS = True to run the experiments")

Available loss experiments (10):
  - loss_arcface: ArcFace (margin=0.5, scale=64)
  - loss_arcface_soft: ArcFace with softer margin (margin=0.3)
  - loss_arcface_hard: ArcFace with harder margin (margin=0.7)
  - loss_subcenter_arcface: SubCenter ArcFace (K=3 subcenters)
  - loss_arcface_triplet: ArcFace + Triplet loss (weight=0.5)
  - loss_triplet_hard: Triplet loss with hard mining
  - loss_triplet_semi_hard: Triplet loss with semi-hard mining
  - loss_cross_entropy: Standard Cross Entropy (baseline)
  - loss_focal: Focal loss (gamma=2.0)
  - loss_random_baseline: Random baseline that returns random embeddings and rankings

Selected 3 experiments to run

Using configuration from notebook:
  - Dataset: JID_Master_Dataset
  - Split field: closed_set_split
  - Backbone: BVRA/MegaDescriptor-L-384
  - Wandb project: camera-trap-reidentification
  - Wandb entity: jaguars

EXPERIMENT 1/3: loss_arcface
Description: ArcFace (margin=0.5, scale=64)
20:36:11 - jid_logger.reidentification.training

train/batch_acc,▁▁▁
train/batch_cls_loss,█▆▁
train/batch_loss,█▆▁
train/batch_acc,0
train/batch_cls_loss,38.78009
train/batch_loss,38.78009


20:36:14 - jid_logger.reidentification.training - INFO - Loading dataset...
20:36:28 - jid_logger.reidentification.training - INFO - Dataset loaded:
20:36:28 - jid_logger.reidentification.training - INFO -   Train: 946 samples
20:36:28 - jid_logger.reidentification.training - INFO -   Val: 120 samples
20:36:28 - jid_logger.reidentification.training - INFO -   Num classes: 76
20:36:28 - jid_logger.reidentification.training - INFO - Using pre-computed embeddings
20:36:28 - jid_logger.reidentification.training - INFO - DataLoaders created:
20:36:28 - jid_logger.reidentification.training - INFO -   Train batches: 30
20:36:28 - jid_logger.reidentification.training - INFO -   Val batches: 4
Model initialized:
  Input dim: 1536
  Hidden dim: 512
  Embedding dim: 256
  Num classes: 76
  ArcFace margin: 0.5
  ArcFace scale: 64.0
  Total parameters: 939,264
20:36:28 - jid_logger.reidentification.training - INFO - Loss: arcface
20:36:28 - jid_logger.reidentification.training - INFO - Training com

✗ loss_arcface failed: cannot access local variable 'compute_validation_map' where it is not associated with a value

EXPERIMENT 2/3: loss_cross_entropy
Description: Standard Cross Entropy (baseline)
20:36:28 - jid_logger.reidentification.training - INFO - Starting re-identification training...
20:36:28 - jid_logger.reidentification.training - INFO - Dataset source: fiftyone
20:36:28 - jid_logger.reidentification.training - INFO - Backbone: BVRA/MegaDescriptor-L-384
20:36:28 - jid_logger.reidentification.training - INFO - Device: cuda
20:36:28 - jid_logger.reidentification.training - INFO - Resource validation passed


train/batch_acc,▁▁▁
train/batch_cls_loss,▁█▁
train/batch_loss,▁█▁
train/batch_acc,0
train/batch_cls_loss,39.22858
train/batch_loss,39.22858


...

✗ loss_cross_entropy failed: cannot access local variable 'compute_validation_map' where it is not associated with a value

EXPERIMENT 3/3: loss_random_baseline
Description: Random baseline that returns random embeddings and rankings
20:36:45 - jid_logger.reidentification.training - INFO - Starting re-identification training...
20:36:45 - jid_logger.reidentification.training - INFO - Dataset source: fiftyone
20:36:45 - jid_logger.reidentification.training - INFO - Backbone: BVRA/MegaDescriptor-L-384
20:36:45 - jid_logger.reidentification.training - INFO - Device: cuda
20:36:45 - jid_logger.reidentification.training - INFO - Resource validation passed


train/batch_acc,▁▁▁
train/batch_cls_loss,█▁▁
train/batch_loss,█▁▁
train/batch_acc,0
train/batch_cls_loss,39.24943
train/batch_loss,39.24943


20:36:48 - jid_logger.reidentification.training - INFO - Loading dataset...
20:37:01 - jid_logger.reidentification.training - INFO - Dataset loaded:
20:37:01 - jid_logger.reidentification.training - INFO -   Train: 946 samples
20:37:01 - jid_logger.reidentification.training - INFO -   Val: 120 samples
20:37:01 - jid_logger.reidentification.training - INFO -   Num classes: 76
20:37:01 - jid_logger.reidentification.training - INFO - Using pre-computed embeddings
20:37:01 - jid_logger.reidentification.training - INFO - Running random-baseline evaluation (no training)


val/map,▁
val/map,0.07243


20:37:03 - jid_logger.reidentification.training - INFO - Random baseline mAP: 0.0724
✓ loss_random_baseline completed: mAP = 0.0724

EXPERIMENT SUMMARY
  loss_arcface: FAILED - cannot access local variable 'compute_validation_map' where it is not associated with a value
  loss_cross_entropy: FAILED - cannot access local variable 'compute_validation_map' where it is not associated with a value
  loss_random_baseline: mAP = 0.0724


## Optional: Load Model for Inference

Load the trained model for making predictions on new data.

In [9]:
# Example: Load model for inference
import torch
from jaguars.reidentification.model import ReidentificationModel
from jaguars.reidentification.backbone import get_backbone

# Load model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize backbone and model
backbone = get_backbone(
    name=config.backbone.name,
    pretrained=config.backbone.pretrained
)

model = ReidentificationModel(
    backbone=backbone,
    embedding_dim=config.model.embedding_dim,
    hidden_dim=config.model.hidden_dim
)

# Load weights
checkpoint = torch.load(model_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

print(f"✓ Model loaded from {model_path}")
print(f"  Trained for {checkpoint.get('epoch', 'N/A')} epochs")
print(f"  Best validation mAP: {checkpoint.get('best_map', 'N/A'):.4f}")

# Now you can use the model for inference
# Example:
# with torch.no_grad():
#     embeddings = model(batch_images)
#     # Compare with gallery embeddings using cosine similarity

ImportError: cannot import name 'ReidentificationModel' from 'jaguars.reidentification.model' (/sc/home/philipp.kolbe/JID/camera-trap-footage/src/jaguars/reidentification/model.py)